<a href="https://colab.research.google.com/github/Mayank12348487487/Week1_Internship_work/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane over the other three because it's the one the starter playground is already built around, and running it first gave me a concrete reason to trust it: on the 30,000-row starter slice, a transparent baseline rule already gets precision@50 = 0.240, and a random forest trained on observable signals pushes that to precision@50 = 0.740 (both numbers from `outputs/model_report.md`, which shipped with this repo — I did not recompute them). That's the core promise of this lane: a learned ranking beating a hand-written rule on a real reviewer queue, with each recommendation carrying a reason code a human can check. It also maps cleanly onto something FlyRank's reviewers already do — deciding which pages to refresh, expand, protect, prune, or monitor — rather than an abstract prediction task. I'm not going freestyle because this lane's data support (44-column starter CSV, 32 clients, plus the warehouse release if I need it later) is already strong enough to answer a real decision without inventing a new one.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [2]:
import pandas as pd

df = pd.read_csv('/content/content_refresh_anonymized.csv')

# A quick sense of the volume behind the decision this project supports
high_demand_declining = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)]
print('Rows in starter data:', len(df))
print('High-demand pages currently trending down (impressions_90d >= 500):', len(high_demand_declining))
print('Share of all rows this represents: {:.1%}'.format(len(high_demand_declining) / len(df)))

Rows in starter data: 30000
High-demand pages currently trending down (impressions_90d >= 500): 9961
Share of all rows this represents: 33.2%


**Decision:** Which content pages should an SEO/content reviewer at FlyRank look at first this week, out of a much larger inventory, when they only have time to check a limited number of pages?

**Who acts on it:** A human content reviewer (or the FlyRank team member who triages the refresh queue). They don't take the ranking as a verdict — they open the top-ranked pages, read the reason codes, and decide the actual action: refresh the content, expand it, protect it, prune it, or just keep monitoring it.

**Cost of a wrong call:** Two different mistakes, and they cost different things:
- *False positive* (the queue flags a page as a priority and it isn't really declining): a reviewer spends limited time on a page that didn't need it, and a page that did need attention waits longer.
- *False negative* (a genuinely declining, high-demand page never surfaces near the top of the queue): the site keeps losing visibility on a page that mattered, and nobody notices until the drop is much worse. The check above shows this isn't a rare edge case — plenty of rows in the starter data already combine real demand (500+ impressions in 90 days) with a downward trend, so a queue that ranks badly has real pages to get wrong.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
declining_rate = (df['trend_direction'] == 'down').mean()
no_position_data = (df['avg_position'] == 0).sum()
median_ctr = df['ctr'].median()  # ctr is a x100 percentage: 0.07 means 0.07%, not 7% -- see docs/data-dictionary.md
n_clients = df['client_id'].nunique()

print(f'Rows: {len(df):,} across {n_clients} pseudonymized clients')
print(f'Declining-label rate (trend_direction == "down"): {declining_rate:.1%}')
print(f'Rows with avg_position == 0 (no position data, not rank zero): {no_position_data:,}')
print(f'Median CTR across all rows: {median_ctr}% (a x100-scaled percentage, not a fraction)')

Rows: 30,000 across 32 pseudonymized clients
Declining-label rate (trend_direction == "down"): 54.2%
Rows with avg_position == 0 (no position data, not rank zero): 1,205
Median CTR across all rows: 0.07% (a x100-scaled percentage, not a fraction)


Three numbers that back this lane:

1. **54.2% of the 30,000 starter rows carry the `down` trend label** — over half the inventory looks like a candidate for review under even the simplest rule, so there's real volume for a ranking model to sort through, not a handful of edge cases.
2. **1,205 rows have `avg_position == 0`**, which the data dictionary says means *no position data*, not first place. A naive score that didn't know this gotcha would rank those rows as if they were ranking #0 — exactly the kind of mistake a careful baseline and a documented data contract is supposed to catch.
3. **Median CTR is 0.07** (i.e. 0.07%, since `ctr` is a x100 percentage in this dataset) — CTR across the inventory is low and skewed, which is why the starter pipeline treats CTR review as its own reason code (`low_ctr_visible_page`, `ctr_review_candidate`) instead of folding it into one blended score.

On top of my own numbers, the pipeline that ships with this repo already ran the full lane once end to end: baseline rules score precision@50 = 0.240 versus random forest precision@50 = 0.740 on a client-holdout split (`outputs/model_report.md`). That's an existing, reproducible result — not something I'm claiming new here — but it's real evidence this lane is worth building on rather than a lane I'm hoping will work out.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [4]:
# The starter label is a proxy, not a future outcome -- worth stating plainly, not just in prose
print('Starter target: is_declining_label = (trend_direction == "down")')
print('This is computed from the CURRENT 90-day window, not a future outcome.')
print('A stronger capstone target: prior 90 days of features -> decline/recovery over the NEXT 30 days.')

Starter target: is_declining_label = (trend_direction == "down")
This is computed from the CURRENT 90-day window, not a future outcome.
A stronger capstone target: prior 90 days of features -> decline/recovery over the NEXT 30 days.


**What this work can say:**
- These are *observed*, trailing-90-day patterns in search and engagement signals — real measurements, not opinions about pages.
- Any relationship between a signal (position, CTR, freshness, word count) and the decline label is *directional* and *observational* — it describes what moved together in this data, not why.
- The final output is *decision-support*: a ranked queue with reason codes that helps a human reviewer spend limited time on the most promising pages first. It is not an automatic publishing or editing decision.
- Confidence tiers (high/medium/low) describe how much evidence backs a recommendation, not a guarantee it's correct.

**What this work will never say:**
- It will never claim a refresh *caused* a recovery — that needs an experiment or a causal design, and this data alone can't provide one.
- It will never claim to have proven or reverse-engineered a Google ranking factor, or to be "predicting Google."
- It won't treat the starter label as ground truth about the future: `is_declining_label` is a proxy built from the current window, so a strong claim would reframe it as a genuinely future-observed outcome before trusting it for real decisions.
- It won't surface anything that looks like a real client name, URL, domain, or query — every id in this data is a pseudonym, and it stays that way in any output.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.